In [ ]:
print("Name: Divyadharshini S\nRoll no: 24BAD022")

!pip install -q transformers torch gradio
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "microsoft/DialoGPT-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()
print("Model loaded successfully!")
print("Model Name:", MODEL_NAME)
print("Execution Device:", device)

prompt = "Hello! How are you?"
input_ids = tokenizer.encode(
    prompt + tokenizer.eos_token,
    return_tensors="pt"
).to(device)
print("\nOriginal Prompt:", prompt)
print("Tokenized Input:", input_ids)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
response_ids = output_ids[:, input_ids.shape[-1]:]
response = tokenizer.decode(
    response_ids[0],
    skip_special_tokens=True
)
print("\nGenerated Chatbot Response:", response)

def generate_response(prompt, history_ids=None):
    new_input_ids = tokenizer.encode(
        prompt + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)
    if history_ids is not None:
        input_ids = torch.cat(
            [history_ids, new_input_ids],
            dim=-1
        )
    else:
        input_ids = new_input_ids
    input_ids = input_ids[:, -800:]
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    new_response_ids = generated_ids[:, input_ids.shape[-1]:]
    answer = tokenizer.decode(
        new_response_ids[0],
        skip_special_tokens=True
    )
    return answer, generated_ids

Name: Divyadharshini S
Roll no: 24BAD022


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Model loaded successfully!
Model Name: microsoft/DialoGPT-small
Execution Device: cpu

Original Prompt: Hello! How are you?
Tokenized Input: tensor([[15496,     0,  1374,   389,   345,    30, 50256]])

Generated Chatbot Response: Hello everyone


In [ ]:
print("Name: Divyadharshini S\nRoll no: 24BAD022")

chat_history_ids = None
def chatbot_response(user_input):
    global chat_history_ids
    response, generated_ids = generate_response(
        user_input,
        chat_history_ids
    )
    chat_history_ids = generated_ids[:, -800:]
    return response
test_queries = [
    "Hello!",
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is deep learning?"
]
chat_history_ids = None
print("CHATBOT FUNCTIONALITY TEST")
for query in test_queries:
    print("\nUser:", query)
    answer = chatbot_response(query)
    print("Chatbot:", answer)

Name: Divyadharshini S
Roll no: 24BAD022
CHATBOT FUNCTIONALITY TEST

User: Hello!
Chatbot: Hi !

User: What is artificial intelligence?
Chatbot: I don't think so , because that would be artificial intelligence .

User: What is machine learning?
Chatbot: I don't think so , because that would be artificial intelligence .

User: What is deep learning?
Chatbot: What is deep learning ?


In [ ]:
print("Name: Divyadharshini S\nRoll no: 24BAD022")

!pip install -q --upgrade gradio
import gradio as gr
import torch
def gradio_chatbot(message, history):
    context_ids = []
    for item in history:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, bot_response = item
            if user_message:
                context_ids.extend(
                    tokenizer.encode(
                        user_message + tokenizer.eos_token,
                        add_special_tokens=False
                    )
                )
            if bot_response:
                context_ids.extend(
                    tokenizer.encode(
                        bot_response + tokenizer.eos_token,
                        add_special_tokens=False
                    )
                )
    current_ids = tokenizer.encode(
        message + tokenizer.eos_token,
        add_special_tokens=False
    )
    all_ids = (context_ids + current_ids)[-800:]
    input_ids = torch.tensor(
        [all_ids],
        dtype=torch.long
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    response_ids = output_ids[:, input_ids.shape[-1]:]
    response = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True
    )
    return response
demo = gr.ChatInterface(
    fn=gradio_chatbot,
    title="Transformer-Based LLM Chatbot",
    description="An interactive chatbot powered by Microsoft DialoGPT-small.",
    examples=[
        "Hello!",
        "What is artificial intelligence?",
        "Explain machine learning."
    ]
)
demo.launch(share=True)


Name: Divyadharshini S
Roll no: 24BAD022
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57e1514fe96bb5e601.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print("Name: Divyadharshini S\nRoll no: 24BAD022")

import time
import os
import zipfile
test_questions = [
    "Hello!",
    "What is artificial intelligence?",
    "Explain machine learning.",
    "What is deep learning?"
]
print("CHATBOT TESTING RESULTS")

for question in test_questions:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()
    answer, _ = generate_response(question)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    print("\nUser:", question)
    print("Chatbot:", answer)
    print("Response Time:", round(end_time - start_time, 2), "seconds")

app_code = r'''
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "microsoft/DialoGPT-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

def chatbot(message, history):

    context_ids = []

    for item in history:
        if item["role"] in ["user", "assistant"]:
            ids = tokenizer.encode(
                item["content"] + tokenizer.eos_token,
                add_special_tokens=False
            )
            context_ids.extend(ids)

    new_ids = tokenizer.encode(
        message + tokenizer.eos_token,
        add_special_tokens=False
    )

    all_ids = (context_ids + new_ids)[-800:]

    input_ids = torch.tensor(
        [all_ids],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response_ids = output_ids[:, input_ids.shape[-1]:]

    response = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True
    )

    return response

demo = gr.ChatInterface(
    fn=chatbot,
    type="messages",
    title="Transformer-Based LLM Chatbot",
    description="Powered by Microsoft DialoGPT-small."
)

if __name__ == "__main__":
    demo.launch()
'''

with open("app.py", "w", encoding="utf-8") as file:
    file.write(app_code)
requirements = """torch
transformers
gradio
safetensors
"""
with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)
with zipfile.ZipFile(
    "chatbot_deployment.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:
    zip_file.write("app.py")
    zip_file.write("requirements.txt")

print(os.listdir())
from google.colab import files
files.download("app.py")
files.download("requirements.txt")
files.download("chatbot_deployment.zip")


Name: Divyadharshini S
Roll no: 24BAD022
CHATBOT TESTING RESULTS

User: Hello!
Chatbot: Hi !
Response Time: 1.52 seconds

User: What is artificial intelligence?
Chatbot: You mean artificial intelligence ?
Response Time: 3.23 seconds

User: Explain machine learning.
Chatbot: What does it mean when you say it's a computer ?
Response Time: 6.62 seconds

User: What is deep learning?
Chatbot: It's a deep learning problem that can be solved with some simple algorithms .
Response Time: 6.59 seconds
['.config', '.gradio', 'chatbot_deployment.zip', 'requirements.txt', 'app.py', 'sample_data']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>